# OCTDL Data Cleaning and Manifest

This notebook audits the OCTDL metadata and images without modifying the raw dataset. It creates a validated manifest for the first AMD versus non-AMD modeling stage.

The validation policy is conservative: an image is usable only when its expected file exists, it can be opened, its dimensions are valid and consistent with the metadata, and its content is not duplicated elsewhere in the manifest.

## 1. Imports and project paths

The project root is discovered from the presence of `data/raw/OCTDL`, so the notebook works when launched from either the repository root or the `notebooks` directory.

In [1]:
from __future__ import annotations

import hashlib
from pathlib import Path

import pandas as pd
from PIL import Image, UnidentifiedImageError


def find_project_root() -> Path:
    """Find the repository root from the expected OCTDL directory."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "raw" / "OCTDL").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find a project root containing data/raw/OCTDL."
    )


PROJECT_ROOT = find_project_root()
OCTDL_ROOT = PROJECT_ROOT / "data" / "raw" / "OCTDL"
LABELS_PATH = OCTDL_ROOT / "OCTDL_labels.csv"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"
MANIFEST_PATH = PROCESSED_ROOT / "manifest.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Labels file: {LABELS_PATH}")
print(f"Labels file exists: {LABELS_PATH.is_file()}")

Project root: C:\Users\vibhu\OneDrive\图片\文档\AMD-Digital-Twin
Labels file: C:\Users\vibhu\OneDrive\图片\文档\AMD-Digital-Twin\data\raw\OCTDL\OCTDL_labels.csv
Labels file exists: True


## 2. Load the OCTDL labels

The original disease labels are retained exactly. The `image_hight` spelling is corrected only in the cleaned in-memory table.

In [2]:
if not LABELS_PATH.is_file():
    raise FileNotFoundError(f"Labels file not found: {LABELS_PATH}")

df = pd.read_csv(LABELS_PATH)
required_columns = {
    "file_name", "disease", "subcategory", "condition",
    "patient_id", "eye", "sex", "year",
    "image_width", "image_hight",
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

print(f"Loaded {len(df):,} metadata rows")
print(f"Columns: {list(df.columns)}")
display(df.head())

Loaded 2,064 metadata rows
Columns: ['file_name', 'disease', 'subcategory', 'condition', 'patient_id', 'eye', 'sex', 'year', 'image_width', 'image_hight']


,file_name,disease,subcategory,condition,patient_id,eye,sex,year,image_width,image_hight
0,amd_1047099_1,AMD,intermediate,MNV_suspected,1047099,0,0,0,1101,410
1,amd_1047099_2,AMD,intermediate,MNV_suspected,1047099,0,0,0,731,265
2,amd_1047099_3,AMD,intermediate,MNV_suspected,1047099,0,0,0,1100,410
3,amd_1047099_4,AMD,intermediate,MNV_suspected,1047099,0,0,0,882,321
4,amd_1084498_1,AMD,late,MNV,1084498,0,0,0,882,321


## 3. Build and verify image paths

The CSV stores image basenames without the `.jpg` suffix. Paths are constructed as `data/raw/OCTDL/<disease>/<file_name>.jpg`.

In [3]:
df["image_path"] = df.apply(
    lambda row: OCTDL_ROOT / str(row["disease"]) / f"{row['file_name']}.jpg",
    axis=1,
)
df["image_exists"] = df["image_path"].map(Path.is_file)

missing_image_files = int((~df["image_exists"]).sum())
print(f"Missing image files: {missing_image_files:,}")
if missing_image_files:
    display(df.loc[~df["image_exists"], ["file_name", "disease", "image_path"]].head())

Missing image files: 0


## 4. Validate readability, dimensions, and image content

Each existing image is checked with Pillow. `verify()` detects truncated or corrupted files, and a second open reads the dimensions. SHA-256 hashes identify duplicate image content without changing the files.

In [4]:
def inspect_image(path: Path) -> dict[str, object]:
    result = {
        "readable": False,
        "actual_width": pd.NA,
        "actual_height": pd.NA,
        "content_hash": pd.NA,
        "read_error": pd.NA,
    }
    try:
        with Image.open(path) as image:
            image.verify()
        with Image.open(path) as image:
            result["actual_width"], result["actual_height"] = image.size
        result["content_hash"] = hashlib.sha256(path.read_bytes()).hexdigest()
        result["readable"] = True
    except (OSError, UnidentifiedImageError, ValueError) as error:
        result["read_error"] = f"{type(error).__name__}: {error}"
    return result

inspection = pd.DataFrame(
    [
        inspect_image(path)
        if path.is_file()
        else {
            "readable": False,
            "actual_width": pd.NA,
            "actual_height": pd.NA,
            "content_hash": pd.NA,
            "read_error": "Missing file",
        }
        for path in df["image_path"]
    ],
    index=df.index,
)
df = pd.concat([df, inspection], axis=1)

df["image_width"] = pd.to_numeric(df["image_width"], errors="coerce")
df["image_hight"] = pd.to_numeric(df["image_hight"], errors="coerce")
df["dimensions_valid"] = (
    df["actual_width"].notna()
    & df["actual_height"].notna()
    & (df["actual_width"] > 0)
    & (df["actual_height"] > 0)
    & (df["image_width"] > 0)
    & (df["image_hight"] > 0)
    & (df["actual_width"] == df["image_width"])
    & (df["actual_height"] == df["image_hight"])
)

df["duplicate_filename"] = df["file_name"].duplicated(keep=False)
df["duplicate_content"] = df["content_hash"].notna() & df["content_hash"].duplicated(keep=False)

print(f"Unreadable or corrupted images: {(~df['readable']).sum():,}")
print(f"Invalid image dimensions: {(~df['dimensions_valid']).sum():,}")
print(f"Duplicate filename rows: {df['duplicate_filename'].sum():,}")
print(f"Duplicate content rows: {df['duplicate_content'].sum():,}")

Unreadable or corrupted images: 0
Invalid image dimensions: 0
Duplicate filename rows: 0
Duplicate content rows: 2


## 5. Audit missing and placeholder metadata

The source uses `0` as a placeholder for some metadata. These values are reported before normalization and are not interpreted as real eye, sex, or year values.

In [5]:
metadata_columns = ["patient_id", "eye", "sex", "year", "image_width", "image_hight"]
placeholder_counts = {}
for column in metadata_columns:
    missing = int(df[column].isna().sum())
    placeholders = int(df[column].astype(str).str.strip().isin(["0", "0.0", "", "nan", "None"]).sum())
    placeholder_counts[column] = {"missing": missing, "zero_or_empty_placeholder": placeholders}

print("Missing and placeholder metadata counts:")
display(pd.DataFrame(placeholder_counts).T)

Missing and placeholder metadata counts:


,missing,zero_or_empty_placeholder
patient_id,0,0
eye,0,1616
sex,0,1616
year,0,1616
image_width,0,0
image_hight,0,0


## 6. Standardize metadata and create the AMD target

Original disease labels are preserved. Unknown categorical values become `Unknown`; unknown years become missing nullable integers.

In [6]:
df = df.rename(columns={"image_hight": "image_height"})

for column in ["eye", "sex"]:
    df[column] = df[column].astype("string").replace({"0": "Unknown", "0.0": "Unknown", "<NA>": "Unknown"})
    df[column] = df[column].fillna("Unknown")

df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df["year"] = df["year"].mask(df["year"] == 0)

df["target_amd"] = (df["disease"] == "AMD").astype("int8")

print("Standardized metadata values:")
for column in ["eye", "sex", "year"]:
    print(f"{column}: {df[column].value_counts(dropna=False).head().to_dict()}")
print(f"AMD target counts: {df['target_amd'].value_counts().to_dict()}")

Standardized metadata values:
eye: {'Unknown': 1616, 'OS': 245, 'OD': 203}
sex: {'Unknown': 1616, 'F': 323, 'M': 125}
year: {<NA>: 1616, np.int64(1948): 36, np.int64(1947): 26, np.int64(1952): 24, np.int64(1946): 24}
AMD target counts: {1: 1231, 0: 833}


## 7. Mark usable records and create the clean manifest

Duplicate content is excluded from `usable` so it cannot silently create leakage in a future train/test split. No raw image is deleted or modified.

In [7]:
df["usable"] = (
    df["image_exists"]
    & df["readable"]
    & df["dimensions_valid"]
    & ~df["duplicate_filename"]
    & ~df["duplicate_content"]
).astype(bool)

manifest_columns = [
    "image_path", "file_name", "patient_id", "disease",
    "subcategory", "condition", "eye", "sex", "year",
    "image_width", "image_height", "target_amd", "usable",
]
manifest = df[manifest_columns].copy()
manifest["image_path"] = manifest["image_path"].map(lambda path: str(path.relative_to(PROJECT_ROOT)))

PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
manifest.to_csv(MANIFEST_PATH, index=False)

print(f"Saved manifest: {MANIFEST_PATH}")
print(f"Manifest columns: {list(manifest.columns)}")
display(manifest.head())

Saved manifest: C:\Users\vibhu\OneDrive\图片\文档\AMD-Digital-Twin\data\processed\manifest.csv
Manifest columns: ['image_path', 'file_name', 'patient_id', 'disease', 'subcategory', 'condition', 'eye', 'sex', 'year', 'image_width', 'image_height', 'target_amd', 'usable']


,image_path,file_name,patient_id,disease,subcategory,condition,eye,sex,year,image_width,image_height,target_amd,usable
0,data\raw\OCTDL\AMD\amd_1047099_1.jpg,amd_1047099_1,1047099,AMD,intermediate,MNV_suspected,Unknown,Unknown,<NA>,1101,410,1,True
1,data\raw\OCTDL\AMD\amd_1047099_2.jpg,amd_1047099_2,1047099,AMD,intermediate,MNV_suspected,Unknown,Unknown,<NA>,731,265,1,True
2,data\raw\OCTDL\AMD\amd_1047099_3.jpg,amd_1047099_3,1047099,AMD,intermediate,MNV_suspected,Unknown,Unknown,<NA>,1100,410,1,True
3,data\raw\OCTDL\AMD\amd_1047099_4.jpg,amd_1047099_4,1047099,AMD,intermediate,MNV_suspected,Unknown,Unknown,<NA>,882,321,1,True
4,data\raw\OCTDL\AMD\amd_1084498_1.jpg,amd_1084498_1,1084498,AMD,late,MNV,Unknown,Unknown,<NA>,882,321,1,True


## 8. Final data quality report

This summary is the checkpoint for the data-preparation phase.

In [8]:
duplicate_images = int(df["duplicate_content"].sum())
quality_report = {
    "Total images": len(df),
    "Total patients": df["patient_id"].nunique(dropna=True),
    "Usable images": int(df["usable"].sum()),
    "AMD images": int(df["target_amd"].sum()),
    "Non-AMD images": int((df["target_amd"] == 0).sum()),
    "Unreadable images": int((~df["readable"]).sum()),
    "Duplicate images": duplicate_images,
    "Missing image files": int((~df["image_exists"]).sum()),
}

print("Final data quality report")
print("-" * 28)
for metric, value in quality_report.items():
    print(f"{metric}: {value:,}")

Final data quality report
----------------------------
Total images: 2,064
Total patients: 820
Usable images: 2,062
AMD images: 1,231
Non-AMD images: 833
Unreadable images: 0
Duplicate images: 2
Missing image files: 0
